# NutritionLLM full run

Trains and tests NutritionLLM on Kaggle's GPU and packs the results into one zip. This is how every
real training run happens, because the project PC has no GPU. The run copies the code from GitHub,
so push your commits first. The two "Model" cells say what a run does; edit those to change the
experiment. A quick try on 40 rows comes first, and the safety stop after it ends the run early if
the code is broken, instead of wasting hours. Look at the printed loss, which should fall, and at
the test tables at the end of each model's cell.

From the project folder: start with `kaggle kernels push -p notebooks/full_run`, watch with
`kaggle kernels logs <your-username>/nutritionllm-full-run --follow`, and collect with
`kaggle kernels output <your-username>/nutritionllm-full-run -p results`. The settings file
`kernel-metadata.json` names the Kaggle account that owns the notebook; to run it under another
account, change the username in its `id` line.

In [ ]:
# Get the code; /kaggle/temp has room for the 16 GB of data, and only /kaggle/working is kept
!mkdir -p /kaggle/temp
%cd /kaggle/temp
!git clone https://github.com/AadiKenwar1/NutritionLLM.git
%cd NutritionLLM

In [ ]:
# Install the tested library versions; Kaggle's old torchao makes peft refuse to load
!pip install -q -r requirements.txt
!pip uninstall -y torchao

In [ ]:
# Download the four datasets and build the training and test files
!python download.py
!python process.py

In [ ]:
# Quick try: train on 40 rows and test 5 photos, to prove the code runs before the long steps
!python train.py --limit 40 --out runs/try
!python test.py --model runs/try --data data/processed/test_real_world.jsonl --limit 5

In [ ]:
# Safety stop: end the run here if the quick try failed, instead of wasting hours
import os
assert os.path.exists("results/try_test_real_world.json"), "The quick try failed."

In [ ]:
# Model A: all photo rows plus the 5,000 USDA typed meals; saves to runs/full_usda
!python train.py
!python test.py --model runs/full_usda --data data/processed/test_real_world.jsonl --limit 0
!python test.py --model runs/full_usda --data data/processed/test_lab.jsonl --limit 0

In [ ]:
# Model B, the comparison run: Nutrition5k only; timeout keeps a slow test from hitting Kaggle's 12-hour limit
!python train.py --mmfood none --no-usda
!timeout 3h python test.py --model runs/none --data data/processed/test_real_world.jsonl --limit 0
!timeout 90m python test.py --model runs/none --data data/processed/test_lab.jsonl --limit 0

In [ ]:
# Pack everything the run made, except the quick try; only /kaggle/working survives the run
!zip -r /kaggle/working/output.zip results runs -x "runs/try/*"